# Day 054 — Exercise 2: Linking Messages to Conversations

**What you'll build:** `add_message(session, conversation_id, role, content)` — insert a message that belongs to a conversation through its **foreign key**.

**Why it matters:** A message is meaningless without its conversation. The `conversation_id` foreign key is the database-level link; the `relationship()` on the models turns that link into Python — `conversation.messages` gives you the list. `add_message` is how the app saves each turn of a chat.

## Provided: Setup + Models + create_schema/create_conversation

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)


def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv

## Your Implementation

In [ ]:
def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Insert a Message linked to conversation_id via its foreign key,
    flush to assign the id, and return it."""
    # TODO: msg = Message(conversation_id=conversation_id, role=role, content=content)
    # TODO: session.add(msg)
    # TODO: session.flush()
    # TODO: return msg
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    engine = memory_engine()
    create_schema(engine)

    # Check 1: add_message assigns an id and the foreign key
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'chat')
            msg = add_message(s, conv.id, 'user', 'hello')
            s.commit()
            assert msg.id is not None, 'message id should be assigned'
            assert msg.conversation_id == conv.id, 'foreign key not set'
        passed += 1; print('✅ Check 1: message linked via conversation_id')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: the relationship exposes the message on the conversation
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'chat2')
            add_message(s, conv.id, 'user', 'hi there')
            s.commit()
            reloaded = s.get(Conversation, conv.id)
            assert len(reloaded.messages) == 1, f'expected 1 message, got {len(reloaded.messages)}'
            assert reloaded.messages[0].content == 'hi there'
        passed += 1; print('✅ Check 2: conversation.messages reflects the insert')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: role and content are stored
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'chat3')
            m = add_message(s, conv.id, 'assistant', 'the answer is 42')
            s.commit()
            assert m.role == 'assistant' and m.content == 'the answer is 42'
        passed += 1; print('✅ Check 3: role + content stored')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: multiple messages accumulate on one conversation
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'chat4')
            add_message(s, conv.id, 'user', 'a')
            add_message(s, conv.id, 'assistant', 'b')
            add_message(s, conv.id, 'user', 'c')
            s.commit()
            reloaded = s.get(Conversation, conv.id)
            assert len(reloaded.messages) == 3, f'expected 3, got {len(reloaded.messages)}'
        passed += 1; print('✅ Check 4: multiple messages accumulate')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: messages of different conversations stay separate
    try:
        with Session(engine) as s:
            c1 = create_conversation(s, 'one')
            c2 = create_conversation(s, 'two')
            add_message(s, c1.id, 'user', 'in one')
            add_message(s, c2.id, 'user', 'in two')
            s.commit()
            assert len(s.get(Conversation, c1.id).messages) == 1
            assert len(s.get(Conversation, c2.id).messages) == 1
        passed += 1; print('✅ Check 5: conversations keep separate messages')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Append a message to a conversation via its foreign key, and flush to assign
    the id. The caller commits."""
    msg = Message(conversation_id=conversation_id, role=role, content=content)
    session.add(msg)
    session.flush()
    return msg
```

**Why this works:** Setting `conversation_id` on the `Message` writes the foreign key that links the two rows in the database. Because the models declare a `relationship()` with `back_populates`, SQLAlchemy keeps the Python side in sync too: after the insert, `conversation.messages` includes the new message. One insert, two views of the same link — the row-level FK and the object-level list.
</details>